# T2SQL Failure Attribution — Log Analysis

Notebook này dùng để:
1. Đọc raw logs JSON từ `output_logs/raw_logs/`
2. Phân tích phân bố root cause
3. Gán nhãn thủ công và export ra `output_logs/annotated_benchmark/`

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm root vào path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

RAW_LOG_DIR = ROOT / 'output_logs' / 'raw_logs'
ANNOTATED_DIR = ROOT / 'output_logs' / 'annotated_benchmark'

print(f'Root: {ROOT}')
print(f'Log files found: {len(list(RAW_LOG_DIR.glob("*.json")))}')

In [ ]:
# ── Load tất cả log files ──────────────────────────────────────────
records = []
for log_file in sorted(RAW_LOG_DIR.glob('*.json')):
    with open(log_file, encoding='utf-8') as f:
        records.append(json.load(f))

df = pd.json_normalize(records)
print(f'Loaded {len(df)} records')
df.head()

In [ ]:
# ── Phân bố Root Cause ────────────────────────────────────────────
if 'annotated_root_cause' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    df['annotated_root_cause'].value_counts().plot(
        kind='barh', ax=ax, color='steelblue'
    )
    ax.set_title('Root Cause Distribution', fontsize=14)
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('Chưa có cột annotated_root_cause — hãy chạy pipeline trước.')

In [ ]:
# ── Gán nhãn thủ công ─────────────────────────────────────────────
# Lọc các record chưa được annotate (root_cause == 'unknown')
if 'annotated_root_cause' in df.columns:
    unknown_df = df[df['annotated_root_cause'] == 'unknown'].copy()
    print(f'Records cần annotate thủ công: {len(unknown_df)}')
    unknown_df[['entry_id', 'x1.question', 'x1.db_id', 'final_sql', 'annotated_root_cause']].head(10)

In [ ]:
# ── Export annotated dataset ──────────────────────────────────────
# Sau khi gán nhãn thủ công, lưu ra annotated_benchmark/

ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

# Ví dụ: export toàn bộ
output_file = ANNOTATED_DIR / 'annotated_dataset.json'
df.to_json(output_file, orient='records', force_ascii=False, indent=2)
print(f'Exported to: {output_file}')